In [ ]:
from datamodule import SyntheticDataModule
from mi_estimator import MIEstimator
from omegaconf import OmegaConf
import pytorch_lightning as pl

# Run a quick test
With this notebook you can play around with the configs and compute MI for different synthetic distributions with infosedd

In [ ]:
pl.seed_everything(42)

seq_length = 2
alphabet_size = 2

infosedd_config = OmegaConf.create({
    "estimator": "infosedd",
    "seq_length": seq_length,
    "batch_size": 1024,
    "alphabet_size": alphabet_size,
    "mutual_information": 0.69,
    "n_generations": 100,
    "n_samples": 10000,
    "sampling_eps": 0.001,
    "graph": "absorb",
    "noise": "loglinear",
    "is_parametric_marginal": False,
    "use_marginal_flag": False,
    "normalize": False,
    "noise_rv": None,
    "variant": "j",
    "dropout": 0,
    "resnet_block_groups": 8,
    "sigma_dim": 2,
    "init_dim": 16,
    "scaling_factor": None,
    "dim_mults": [
        1,
        1
    ]
})

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=1000,
    check_val_every_n_epoch=10,
    logger=False,
    enable_progress_bar=True,
    limit_val_batches=1.0,
)

data_module = SyntheticDataModule(infosedd_config)
model = MIEstimator(infosedd_config)

trainer.fit(model, data_module)


In [ ]:
import torch
import time
!export CUDA_VISIBLE_DEVICES=4
with torch.no_grad():
    start = time.time()
    trainer.validate(model, datamodule=data_module)
    end = time.time()
    print(f"Validation time: {end - start:.2f} seconds")

## Using Your Own CSV Dataset

To use your own CSV file with the InfoSEDD estimator:

### 1. Prepare your CSV file
- Two columns containing text data (no headers required)
- Example format:
```
hello world,goodbye world
foo bar,baz qux
text1,text2
```

### 2. Update the configuration
- Set `file_path` to your CSV file path
- Set `x_col: 0` and `y_col: 1` for the column indices (not lists!)
- The system will automatically:
  - Build a character-level vocabulary from your text
  - Set `alphabet_size` and `seq_length` based on the data

### 3. Key parameters to adjust
- `batch_size`: Adjust based on your dataset size and GPU memory
- `max_epochs`: Increase for better convergence on real data
- `check_val_every_n_epoch`: How often to validate during training

### 4. Example with your own data
```python
config = OmegaConf.create({
    "estimator": "infosedd",
    "file_path": "/path/to/your/dataset.csv",
    "x_col": 0,  # First column
    "y_col": 1,  # Second column
    "batch_size": 512,
    "max_epochs": 2000,
    # ... other parameters
})
```

The tokenizer maps each unique column to a unique value, not suitable for long, diverse sequential data like text

In [ ]:
from datamodule import CSVDataModule
from mi_estimator import MIEstimator
from omegaconf import OmegaConf
import pytorch_lightning as pl

In [ ]:
pl.seed_everything(42)

infosedd_config = OmegaConf.create({
    "estimator": "infosedd",
    "delimiter": ",",
    "batch_size": 1024,
    "file_path": "/home/foresti/mutinfo-diffusion/test2.csv",
    "sampling_eps": 0.001,
    "graph": "absorb",
    "noise": "loglinear",
    "is_parametric_marginal": False,
    "use_marginal_flag": False,
    "normalize": False,
    "noise_rv": None,
    "variant": "j",
    "dropout": 0,
    "x_col": [0],
    "y_col": [1],
    "resnet_block_groups": 8,
    "sigma_dim": 2,
    "init_dim": 16,
    "scaling_factor": None,
    "dim_mults": [
        1,
        1
    ]
})

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=1000,
    check_val_every_n_epoch=10,
    logger=False,
    enable_progress_bar=True,
    limit_val_batches=1.0,
)

data_module = CSVDataModule(infosedd_config)
data_module.setup()
infosedd_config.alphabet_size = data_module.alphabet_size
infosedd_config.seq_length = data_module.seq_length
model = MIEstimator(infosedd_config)

trainer.fit(model, data_module)
